In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

from docling.document_converter import DocumentConverter
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PAPER_DIR = PROJECT_ROOT / "data" / "papers"

In [ ]:
import re

def get_section_level(heading: str) -> int:
    match = re.match(r"^(\d+(?:\.\d+)*)\s+", heading)

    if not match:
        return 1

    section_number = match.group(1)

    return section_number.count(".") + 1

def get_section_number(heading: str):
    match = re.match(r"^(\d+(?:\.\d+)*)\s+", heading)

    if match:
        return match.group(1)

    return None

@dataclass
class Section:
    heading: str
    section_number: Optional[str]
    level: int
    page_start: Optional[int]
    page_end: Optional[int]
    text: str


@dataclass
class Paper:
    paper_id: str
    title: str
    authors: List[str]
    year: int
    source_file: str
    sections: List[Section] = field(default_factory=list)

def build_sections(doc):
    sections = []
    current_section = None

    for item, level in doc.iterate_items():
        item_type = type(item).__name__
        text = getattr(item, "text", "").strip()

        if not text:
            continue

        prov = getattr(item, "prov", [])
        page_no = prov[0].page_no if prov else None

        if "SectionHeader" in item_type:
            current_section = {
                "heading": text,
                "section_number": get_section_number(text),
                "level": get_section_level(text),
                "pages": [],
                "text_parts": []
            }

            if page_no is not None:
                current_section["pages"].append(page_no)

            sections.append(current_section)

        elif current_section is not None:
            current_section["text_parts"].append(text)

            if page_no is not None:
                current_section["pages"].append(page_no)

    section_objects = []

    for section in sections:
        text = "\n\n".join(section["text_parts"])

        page_start = min(section["pages"]) if section["pages"] else None
        page_end = max(section["pages"]) if section["pages"] else None

        section_objects.append(
            Section(
                heading=section["heading"],
                section_number=section["section_number"],
                level=section["level"],
                page_start=page_start,
                page_end=page_end,
                text=text
            )
        )

    return section_objects

def build_paper(
    doc,
    paper_id: str,
    title: str,
    authors: list[str],
    year: int,
    source_file: str
):
    sections = build_sections(doc)

    return Paper(
        paper_id=paper_id,
        title=title,
        authors=authors,
        year=year,
        source_file=source_file,
        sections=sections
    )

paper_metadata = {
    "01_rag_lewis_2020.pdf": {
        "paper_id": "rag_lewis_2020",
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "authors": [],
        "year": 2020,
    },

    "02_dpr_karpukhin_2020.pdf": {
        "paper_id": "dpr_karpukhin_2020",
        "title": "Dense Passage Retrieval for Open-Domain Question Answering",
        "authors": [],
        "year": 2020,
    },

    "03_colbert_khattab_2020.pdf": {
        "paper_id": "colbert_khattab_2020",
        "title": "Efficient and Effective Passage Search via Contextualized Late Interaction over BERT",
        "authors": [],
        "year": 2020,
    },    

    "04_hyde_gao_2023.pdf": {
        "paper_id": "hyde_gao_2023",
        "title": "Precise Zero-Shot Dense Retrieval without Relevance Labels",
        "authors": [],
        "year": 2023,
    }, 

     "05_rag_survey_gao_2023.pdf": {
        "paper_id": "rag_survey_gao_2023",
        "title": "Retrieval-Augmented Generation for Large Language Models: A Survey",
        "authors": [],
        "year": 2023,
    },

    "06_self_rag_asai_2023.pdf": {
        "paper_id": "self_rag_asai_2023",
        "title": "SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE THROUGH SELF-REFLECTION",
        "authors": [],
        "year": 2023,
    },

    "07_lost_in_middle_liu_2023.pdf": {
        "paper_id": "lost_in_middle_liu_2023",
        "title": "Lost in the Middle: How Language Models Use Long Contexts",
        "authors": [],
        "year": 2023,
    },    


    "08_raptor_sarthi_2024.pdf": {
        "paper_id": "raptor_sarthi_2024",
        "title": "RAPTOR: RECURSIVE ABSTRACTIVE PROCESSING FOR TREE-ORGANIZED RETRIEVAL",
        "authors": [],
        "year": 2024,
    },    


    "09_crag_yan_2024.pdf": {
        "paper_id": "crag_yan_2024",
        "title": "Corrective Retrieval Augmented Generation",
        "authors": [],
        "year": 2024,
    },    

    "10_qwen3_embedding_2025.pdf": {
        "paper_id": "qwen3_embedding_2025",
        "title": "Qwen3 Embedding: Advancing Text Embedding and Reranking Through Foundation Models",
        "authors": [],
        "year": 2025,
    }
}


@dataclass
class Chunk:
    chunk_id: str
    paper_id: str
    title: str
    section_heading: str
    section_number: Optional[str]
    page_start: Optional[int]
    page_end: Optional[int]
    chunk_index: int
    text: str

    
def split_text_with_overlap(
    text: str,
    chunk_size: int = 600,
    overlap: int = 100
):
    words = text.split()

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk_words = words[start:end]
        chunk_text = " ".join(chunk_words)

        chunks.append(chunk_text)

        start += chunk_size - overlap

    return chunks

In [ ]:
converter = DocumentConverter()
papers = []

for file in PAPER_DIR.glob("*.pdf"):
    metadata = paper_metadata[file.name]

    result = converter.convert(file)
    doc = result.document

    paper = build_paper(
        doc=doc,
        paper_id=metadata["paper_id"],
        title=metadata["title"],
        authors=metadata["authors"],
        year=metadata["year"],
        source_file=file.name
    )

    papers.append(paper)

    print(
        f"Built: {paper.paper_id} | "
        f"sections: {len(paper.sections)}"
    )

In [ ]:
section = papers[0].sections[1]

text_chunks = split_text_with_overlap(section.text)

print("Section:", section.heading)
print("Number of chunks:", len(text_chunks))
print()
print(text_chunks[0][:1000])

In [ ]:
def slugify(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def chunk_section(
    paper: Paper,
    section: Section,
    section_index: int,
    chunk_size: int = 600,
    overlap: int = 100
):
    text_chunks = split_text_with_overlap(
        section.text,
        chunk_size=chunk_size,
        overlap=overlap
    )

    chunks = []
    section_slug = slugify(section.heading)

    for index, chunk_text in enumerate(text_chunks):
        chunk = Chunk(
            chunk_id=(
                f"{paper.paper_id}_"
                f"s{section_index}_"
                f"{section_slug}_"
                f"{index}"
            ),
            paper_id=paper.paper_id,
            title=paper.title,
            section_heading=section.heading,
            section_number=section.section_number,
            page_start=section.page_start,
            page_end=section.page_end,
            chunk_index=index,
            text=chunk_text
        )

        chunks.append(chunk)

    return chunks

In [ ]:
all_chunks = []

for paper in papers:
    for section_index, section in enumerate(paper.sections):
        section_chunks = chunk_section(
            paper=paper,
            section=section,
            section_index=section_index,
            chunk_size=600,
            overlap=100
        )

        all_chunks.extend(section_chunks)

print("Total chunks:", len(all_chunks))

In [ ]:
from collections import Counter

chunk_counts = Counter(chunk.paper_id for chunk in all_chunks)

for paper_id, count in chunk_counts.items():
    print(paper_id, "->", count)

In [ ]:
chunk_lengths = [len(chunk.text.split()) for chunk in all_chunks]

print("Total chunks:", len(chunk_lengths))
print("Minimum words:", min(chunk_lengths))
print("Maximum words:", max(chunk_lengths))
print("Average words:", sum(chunk_lengths) / len(chunk_lengths))

In [ ]:
small_chunks = [
    chunk
    for chunk in all_chunks
    if len(chunk.text.split()) < 100
]

print("Small chunks:", len(small_chunks))

In [ ]:
for chunk in small_chunks[:10]:
    print("ID:", chunk.chunk_id)
    print("Section:", chunk.section_heading)
    print("Words:", len(chunk.text.split()))
    print("Text:", chunk.text[:500])
    print("-" * 80)

In [ ]:
chunk_ids = [chunk.chunk_id for chunk in all_chunks]

print("Total chunks:", len(chunk_ids))
print("Unique chunk IDs:", len(set(chunk_ids)))
print("Duplicates:", len(chunk_ids) - len(set(chunk_ids)))

In [ ]:
from collections import Counter

counts = Counter(chunk.chunk_id for chunk in all_chunks)

for chunk_id, count in counts.items():
    if count > 1:
        print(chunk_id, "->", count)

In [ ]:
from dataclasses import asdict
import json

PROCESSED_DIR = PROJECT_ROOT /'data'/'processed'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

chunks_path = PROCESSED_DIR / "chunks.json"

with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(
        [asdict(chunk) for chunk in all_chunks],
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Saved {len(all_chunks)} chunks to {chunks_path}")